Only standard-library modules and Flask are used, and there is no `input()`.

## Popcorn Hack 1: read the API, then use it (changed version)

**What I changed:** the jersey range is now 1-55, and the court is a 196 square foot court instead of 144.

**Prediction (before running):** the jersey is a random whole number from 1 to 55, including both ends; the court side is `math.sqrt(196)` = `14.0` feet; the average is 122 / 5 = 24.4, which `math.ceil` rounds up to 25. The random game is one of the five point totals.

In [ ]:
# CODE_RUNNER: Popcorn 1 - Read the API, then use it

import math
import random

# Read the API before calling a procedure
print("random.randint:", random.randint.__doc__)
print("math.ceil:", math.ceil.__doc__)
print()

points_per_game = [24, 18, 31, 27, 22]

# random.choice picks one item out of a list
print("Random game to review:", random.choice(points_per_game), "points")

# changed: jerseys now go from 1 to 55 (both ends included)
jersey = random.randint(1, 55)
print("Random jersey number:", jersey)

# changed: a 196 square foot court
side = math.sqrt(196)
print("Square court side:", side, "feet")

# math.ceil rounds up to the next whole number
average = sum(points_per_game) / len(points_per_game)
rounded_up = math.ceil(average)
print("Points per game:", average, "-> rounded up:", rounded_up)

**Actual output (one real run; the random lines change every run):**

```text
random.randint: Return random integer in range [a, b], including both end points.
        
math.ceil: Return the ceiling of x as an Integral.

This is the smallest integer >= x.

Random game to review: 18 points
Random jersey number: 40
Square court side: 14.0 feet
Points per game: 24.4 -> rounded up: 25
```

Matched: the side is 14.0 and the average rounds up to 25. The jersey stayed within 1-55.

## Popcorn Hack 2: write a Flask route (changed version)

**What I changed:** my own roster, plus a new `/record` route defined above `app.test_client()`.

In [ ]:
# CODE_RUNNER: Popcorn 2 - Write a Flask Route

from flask import Flask, jsonify

app = Flask(__name__)

# A route is a procedure with a web address attached to it
@app.route("/score")
def score():
    return jsonify({"home": 58, "away": 52, "quarter": 3})

@app.route("/roster")
def roster():
    return jsonify({"players": ["Jordan", "Maya", "Leo", "Priya"]})

# My own route: the team's win/loss record
@app.route("/record")
def record():
    return jsonify({"wins": 7, "losses": 3})

# test_client visits the addresses without needing a browser
client = app.test_client()

print("/score says:", client.get("/score").get_json())
print("/roster says:", client.get("/roster").get_json())
print("/record says:", client.get("/record").get_json())

**Actual output:**

```text
/score says: {'away': 52, 'home': 58, 'quarter': 3}
/roster says: {'players': ['Jordan', 'Maya', 'Leo', 'Priya']}
/record says: {'losses': 3, 'wins': 7}
```

## Homework: Season Stats Report

My own sport (volleyball), my own five matches and my own next match date (Oct. 14, 2026). Four standard-library imports: `json`, `math`, `statistics`, `datetime`. `summarize_season` has a docstring giving its Parameters and Returns.

In [ ]:
# CODE_RUNNER: Homework - Season Stats Report

import json
import math
import statistics
from datetime import date

# My sport: volleyball. "points" = points I scored in the match (kills + aces + blocks).
matches = [
    {"date": "2026-09-08", "opponent": "Westview", "points": 11, "minutes": 70},
    {"date": "2026-09-12", "opponent": "Del Norte", "points": 7, "minutes": 55},
    {"date": "2026-09-17", "opponent": "Westview", "points": 14, "minutes": 82},
    {"date": "2026-09-22", "opponent": "Mt. Carmel", "points": 9, "minutes": 64},
    {"date": "2026-09-26", "opponent": "Rancho Bernardo", "points": 16, "minutes": 90},
]

def summarize_season(season):
    """
    Summarize a season of matches.

    Parameters:
        season: list of dictionaries, each with "date", "opponent", "points", and "minutes"
    Returns:
        dictionary with total_points, average_points, best_game,
        points_by_opponent, and points_per_hour (rounded up)
    """
    points = []
    total_minutes = 0
    points_by_opponent = {}
    for match in season:
        points.append(match["points"])
        total_minutes = total_minutes + match["minutes"]
        opponent = match["opponent"]
        points_by_opponent[opponent] = points_by_opponent.get(opponent, 0) + match["points"]

    total_points = sum(points)
    return {
        "total_points": total_points,
        "average_points": round(statistics.mean(points), 1),
        "best_game": max(points),
        "points_by_opponent": points_by_opponent,
        "points_per_hour": math.ceil(total_points / (total_minutes / 60)),
    }

report = summarize_season(matches)

# date - date gives a timedelta; .days is the whole number of days between them
next_match = date(2026, 10, 14)
report["days_until_next_game"] = (next_match - date.today()).days

print(json.dumps(report, indent=2))

# requirements.txt lists only packages installed with pip.
# json, math, statistics and datetime are NOT listed: they are part of the
# standard library and ship with every Python install.
requirements_txt = """pandas
Flask
"""
print()
print("requirements.txt:")
print(requirements_txt)

**Actual output (run on 2026-09-30):**

```text
{
  "total_points": 57,
  "average_points": 11.4,
  "best_game": 16,
  "points_by_opponent": {
    "Westview": 25,
    "Del Norte": 7,
    "Mt. Carmel": 9,
    "Rancho Bernardo": 16
  },
  "points_per_hour": 10,
  "days_until_next_game": 14
}

requirements.txt:
pandas
Flask
```

## Submission Notes

```text
Lesson: CSP 3.14 Libraries
Popcorn 1: what I changed = jersey range 1-55 and a 196 sq ft court (sqrt = 14.0), with a prediction written first
Popcorn 2: my extra route = /record -> {"wins": 7, "losses": 3}
Homework: libraries used = json, statistics, datetime (plus math)
Homework: days until my next game = 14 (run on 2026-09-30)
```

Notebook: https://rydenb.github.io/portfolio/python/libraries-hw